# Day 84: Speech-to-Speech Pipeline Latency

**Layer:** Modalities


## Concept Overview

A speech-to-speech pipeline chains ASR (Whisper) -> LLM -> TTS. Total latency is the sum of all three stages plus network hops. Streaming each stage concurrently can reduce perceived latency by 40-60%.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Pipeline Latency Budget

Model each stage independently, then compute total and streaming latency.


In [ ]:
import numpy as np

def speech_to_speech_latency(audio_s, prompt_len=200, output_len=100):
    """Estimate end-to-end speech-to-speech latency."""
    # ASR: Whisper small, RTF=0.1
    asr_ms = audio_s * 0.1 * 1000
    # LLM: TTFT + decode
    ttft_ms = prompt_len * 0.08
    decode_ms = output_len * 20  # 20ms/token decode
    # TTS: ~50ms TTFA for streaming
    tts_ms = output_len * 3  # 3ms/token synthesis
    
    sequential = asr_ms + ttft_ms + decode_ms + tts_ms
    streaming = asr_ms + ttft_ms + 50  # TTFA while decode+TTS stream
    
    return {
        "asr_ms": asr_ms,
        "ttft_ms": ttft_ms,
        "decode_ms": decode_ms,
        "tts_ms": tts_ms,
        "sequential_ms": sequential,
        "streaming_ttfa_ms": streaming
    }

print("Speech-to-speech latency breakdown:")
print(f"{'Component':<20} {'5s audio':>12} {'10s audio':>12}")
for audio in [5.0, 10.0]:
    r = speech_to_speech_latency(audio)
    print(f"  audio={audio}s")
    for k,v in r.items():
        print(f"  {k:<20} {v:>12.0f}ms")
    print()
print("Streaming: pipeline stages run concurrently -> TTFA is the critical metric.")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- A speech-to-speech pipeline chains ASR (Whisper) -> LLM -> TTS.
- A speech-to-speech pipeline chains ASR (Whisper) -> LLM -> TTS. Total latency is....
- Day 84 implementation complete.
